# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Haryomhidhe/Machine-learning-flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [79]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"SELECT DISTINCT regexp_extract(file, 'month=[0-9-]+') AS month FROM glob('{rel}/fact_content_daily_performance/*/*.parquet') ORDER BY month").show()

┌───────────────┐
│     month     │
│    varchar    │
├───────────────┤
│ month=2025-01 │
│ month=2025-02 │
│ month=2025-03 │
│ month=2025-04 │
│ month=2025-05 │
│ month=2025-06 │
│ month=2025-07 │
│ month=2025-08 │
│ month=2025-09 │
│ month=2025-10 │
│ month=2025-11 │
│ month=2025-12 │
│ month=2026-01 │
│ month=2026-02 │
│ month=2026-03 │
│ month=2026-04 │
│ month=2026-05 │
│ month=2026-06 │
├───────────────┤
│    18 rows    │
└───────────────┘



In [80]:
feb = con.sql(f"""
SELECT content_hash_id,
  SUM(gsc_impressions) AS imps,
  SUM(gsc_clicks) AS clicks,
  AVG(gsc_avg_position) AS pos
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

base = feb[feb['imps'] > 100].copy()
base['pos_bucket'] = pd.cut(base['pos'], [0, 2, 3, 5, 10, 20, 50, 10000],
                            labels=['1-2', '2-3', '3-5', '5-10', '10-20', '20-50', '50+'])
expected = base.groupby('pos_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    clicks=('clicks', 'sum'),
    imps=('imps', 'sum'))
expected['expected_ctr_pct'] = expected['clicks'] / expected['imps'] * 100
expected

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n,clicks,imps,expected_ctr_pct
pos_bucket,,,,
1-2,4839,41847.0,11866419.0,0.352651
2-3,6235,94953.0,21542278.0,0.440775
3-5,14591,189199.0,51589704.0,0.366738
5-10,26394,154377.0,50888000.0,0.303366
10-20,16706,69201.0,22252028.0,0.310987
20-50,10302,30317.0,19883322.0,0.152475
50+,1056,174.0,366912.0,0.047423


In [81]:
base['ctr'] = base['clicks'] / base['imps'] * 100
base.groupby('pos_bucket', observed=True)['ctr'].agg(
    n='count', median_ctr='median', p75=lambda s: s.quantile(0.75))

,n,median_ctr,p75
pos_bucket,,,
1-2,4839,0.161812,0.376530
2-3,6235,0.248822,0.509431
3-5,14591,0.224803,0.447761
5-10,26394,0.160508,0.402901
10-20,16706,0.073046,0.335219
20-50,10302,0.000000,0.170322
50+,1056,0.000000,0.000000


In [82]:
def month_totals(m):
    return con.sql(f"""
    SELECT content_hash_id,
      SUM(gsc_impressions) AS imps,
      SUM(gsc_clicks) AS clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month={m}/*.parquet')
    GROUP BY content_hash_id
    """).df()

feb = month_totals('2026-02')
mar = month_totals('2026-03')
d = feb.merge(mar, on='content_hash_id', suffixes=('_feb', '_mar'))
d['clicks_mar_adj'] = d['clicks_mar'] * 28 / 31
d['imps_mar_adj'] = d['imps_mar'] * 28 / 31

for min_clicks in [10, 20, 50]:
    s = d[d['clicks_feb'] >= min_clicks]
    fell = s['clicks_mar_adj'] < 0.7 * s['clicks_feb']
    held = s['imps_mar_adj'] >= 0.8 * s['imps_feb']
    print("min Feb clicks", min_clicks, "| pages:", len(s),
          "| label=1:", round((fell & held).mean() * 100, 1), "%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

min Feb clicks 10 | pages: 13336 | label=1: 16.1 %
min Feb clicks 20 | pages: 6478 | label=1: 15.1 %
min Feb clicks 50 | pages: 2104 | label=1: 11.8 %


In [83]:
feb_pos = con.sql(f"""
SELECT content_hash_id, AVG(gsc_avg_position) AS pos
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

data = d.merge(feb_pos, on='content_hash_id')
data = data[data['clicks_feb'] >= 20].copy()

data['label'] = ((data['clicks_mar_adj'] < 0.7 * data['clicks_feb']) &
                 (data['imps_mar_adj'] >= 0.8 * data['imps_feb'])).astype(int)

data['ctr_feb'] = data['clicks_feb'] / data['imps_feb'] * 100
severity = 2 * (data['ctr_feb'] < 0.15) + 1 * (data['pos'] > 10)
data['baseline_score'] = (data['imps_feb'] * severity).where(data['imps_feb'] > 100, 0)

print("pages:", len(data))
print("label=1 share:", round(data['label'].mean() * 100, 1), "%")
print("pages where baseline score > 0:", (data['baseline_score'] > 0).sum())
print("pages with pos < 1:", (data['pos'] < 1).sum())

pages: 6478
label=1 share: 15.1 %
pages where baseline score > 0: 1388
pages with pos < 1: 41


In [84]:
bad_ids = data.loc[data['pos'] < 1, ['content_hash_id']]
con.register('bad_ids', bad_ids)

print(con.sql(f"""
SELECT COUNT(*) AS days,
       SUM((gsc_avg_position = 0)::INT) AS zero_days,
       SUM((gsc_avg_position IS NULL)::INT) AS null_days
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
JOIN bad_ids USING (content_hash_id)
""").df())

print(data.loc[data['pos'] < 1, 'pos'].describe())

   days  zero_days  null_days
0  1148        2.0       15.0
count    41.000000
mean      0.761375
std       0.153168
min       0.314707
25%       0.643797
50%       0.783187
75%       0.868165
max       0.977644
Name: pos, dtype: float64


In [85]:
print(con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
""").df()[['column_name', 'column_type']])

print(con.sql(f"""
SELECT DISTINCT regexp_extract(file, 'internship-warehouse/([^/]+)/', 1) AS table_folder
FROM glob('{rel}/**/*.parquet')
""").df())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [86]:
clients = con.sql(f"""
SELECT content_hash_id,
       COUNT(DISTINCT client_hash_id) AS n_clients,
       MIN(client_hash_id) AS client_hash_id
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

data = data.merge(clients, on='content_hash_id')

print("pages:", len(data))
print("pages linked to more than one client:", (data['n_clients'] > 1).sum())
print("number of clients:", data['client_hash_id'].nunique())
print(data['client_hash_id'].value_counts().head(5))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages: 6478
pages linked to more than one client: 0
number of clients: 23
client_hash_id
client_73cda7b4e4f265ea    2003
client_62f4a7e64f5e0096    1432
client_23a62021009f63c4    1321
client_e547b89c05043229     641
client_fef1a8f436438636     395
Name: count, dtype: int64


In [87]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
data = data.reset_index(drop=True)
data['fold'] = -1

for k, (train_idx, test_idx) in enumerate(gkf.split(data, data['label'], groups=data['client_hash_id'])):
    data.loc[test_idx, 'fold'] = k

print(data.groupby('fold').agg(
    pages=('label', 'size'),
    clients=('client_hash_id', 'nunique'),
    label_rate=('label', 'mean')
).round(3))

      pages  clients  label_rate
fold                            
0      2003        1       0.144
1      1432        1       0.188
2      1321        1       0.153
3       861        4       0.111
4       861       16       0.141


In [88]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

data['log_imps'] = np.log1p(data['imps_feb'])
data['log_clicks'] = np.log1p(data['clicks_feb'])
features = ['log_imps', 'log_clicks', 'pos', 'ctr_feb']

def prec_at_k(y, score, k=50, seed=0):
    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(len(score)), -score))
    return y[order[:k]].mean()

rows = []
for k in range(5):
    tr = data[data['fold'] != k]
    te = data[data['fold'] == k]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(tr[features], tr['label'])
    p = model.predict_proba(te[features])[:, 1]
    y = te['label'].values
    rows.append({
        'fold': k,
        'base_rate': y.mean(),
        'model': prec_at_k(y, p),
        'baseline': prec_at_k(y, te['baseline_score'].values),
        'random': np.mean([prec_at_k(y, np.random.default_rng(s).random(len(te)), seed=s) for s in range(200)])
    })

res = pd.DataFrame(rows).round(3)
print(res)
print(res[['model', 'baseline', 'random']].mean().round(3))

   fold  base_rate  model  baseline  random
0     0      0.144   0.20      0.08   0.144
1     1      0.188   0.42      0.06   0.186
2     2      0.153   0.16      0.16   0.158
3     3      0.111   0.16      0.10   0.108
4     4      0.141   0.10      0.10   0.141
model       0.208
baseline    0.100
random      0.147
dtype: float64


In [89]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(data[features], data['label'])
coefs = pd.Series(model[-1].coef_[0], index=features).round(3)
print(coefs.sort_values())

log_imps     -0.301
ctr_feb      -0.289
log_clicks    0.035
pos           0.046
dtype: float64


In [90]:
rows = []
for k in range(5):
    tr = data[data['fold'] != k]
    te = data[(data['fold'] == k) & (data['clicks_feb'] >= 50)]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(tr[features], tr['label'])
    p = model.predict_proba(te[features])[:, 1]
    y = te['label'].values
    rows.append({'fold': k, 'pages': len(te), 'base_rate': y.mean(),
                 'model': prec_at_k(y, p), 'baseline': prec_at_k(y, te['baseline_score'].values)})
print(pd.DataFrame(rows).round(3))

   fold  pages  base_rate  model  baseline
0     0    705      0.104   0.08      0.06
1     1    484      0.151   0.16      0.14
2     2    393      0.117   0.12      0.08
3     3    291      0.096   0.14      0.14
4     4    231      0.126   0.16      0.14


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [91]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

ML-08 Section 1: Method choice (draft)
Draft placeholder. Rewrite in your own words before submitting.
Method choice
My lane is refresh and opportunity scoring. The question is which pages should be refreshed first. For this I used Logistic Regression to predict whether a page's clicks will fall in the next month, and I ranked pages by that predicted chance.
The label is 1 if a page's March clicks (scaled by 28/31 to match February's length) fell below 70% of February's while its March impressions stayed at 80% or more of February's. The inputs come only from February 2026, and the label comes only from March 2026, so there is no leakage. I only judged pages with at least 20 clicks in February, because pages with fewer clicks make a very noisy label.
I chose Logistic Regression over a Random Forest or Gradient Boosting for three reasons. First, it is simple and its weights can be read, so I can explain which inputs push a page toward "clicks will fall". Second, the label is noisy and there are only 23 clients, so a more complex model is more likely to learn noise than real patterns. Third, the card says not to reward complexity alone, and a plain model compared honestly against my Week 4 baseline is more useful than a complex one that wins by accident.
The inputs are log impressions, log clicks, average position and CTR, all from February. I used the log of impressions and clicks because those counts are very skewed, and I scaled all four inputs so the weights are comparable.
Limits to state
• Zero-click pages cannot be labelled, so the zero-click picks from my Week 4 rule are not scored here.
• Pages with few clicks make a noisy label, so I set the minimum at 20 February clicks.
• The label measures "got worse", not "is bad".
• 41 pages have an average position below 1, which is impossible for a real position. I kept them and treated them as top ranked.
• Only pages that appear in both February and March can be labelled.

In [92]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [93]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.